# **Experiment Notebook**



---
## Setup Environment

In [1]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT1",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 16.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
hdbscan 0.8.41 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
umap-learn 0.5.11 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Mounted at /content/gdrive

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT1/data


---
## Student Information

In [2]:
# <Student to fill this section and then remove this comment>
student_name = "Rose Marie Tazbaz"
student_id = "25742507"

In [3]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [4]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [5]:
import numpy as np

### 0.b Import Packages

In [6]:
# DO NOT MODIFY THE CODE IN THIS CELL
import pandas as pd
import altair as alt

---
## A. Experiment Description

In [7]:
# DO NOT MODIFY THE CODE IN THIS CELL
experiment_id = "2"
print_tile(size="h1", key='experiment_id', value=experiment_id)

In [8]:
# <Student to fill this section and then remove this comment>
experiment_hypothesis = "In this experiment, I want to test whether elastic net regression can improve the prediction accuracy compared to the linear regression model used in the previous experiment. Elastic net combines two types of regularization (L1 and L2), which can help reduce overfitting and improve the stability of the model when there are many correlated features. In the pricing context, many features may be related to each other. For example, engine capacity, fuel consumption, and engine cylinders are technically connected. Also, brand and model can also be correlated. Because of these possible relationships, a simple linear regression model could rely too much on certain variables. Elastic net is worth testing because it can reduce the importance of less useful features while keeping the most relevant ones. This could help the model focus on the characteristics that truly influence resale prices, such as vehicle age, engine specifications, and kilometres driven. From a business perspective, improving the accuracy of the price estimation model is important because the car re-seller wants to provide customers with realistic price expectations when they bring their cars for resale. Better predictions could increase customer trust in the system and help the dealership price vehicles more efficiently."

In [9]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='experiment_hypothesis', value=experiment_hypothesis)

In [10]:
# <Student to fill this section and then remove this comment>
experiment_expectations = "From this experiment, I expect that it will slightly improve the prediction performance compared to the basic linear regression model. As elastic net applies regularization, it may reduce noise in the model and improve generalization when predicting unseen vehicles (not in the training dataset). The idea is that model would produce lower prediction errors, meaning a lower MAE and RMSE than the previous experiment, while maintaining/improving the R2 score. The goal would be to reduce the average prediction error by a small but meaningful amount compared to the current MAE of around $9,600. There are many different possible outcomes, for example, the best case scenario would be that this model improves prediction accuracy and produces lower MAE and RMSE values. This would suggest that regularization helps the model handle correlated features and improve price predictions. It could also happen that the performance remains similar to linear regression, that would indicate that regularization does not change the model performance significantly, at least for this dataset. As worst case scenario, the model would perform worse because of excessive regularization, which could remove useful information from the model. Regardless of the result, this experiment will provide useful insight into whether adding regularization helps improve the reliability of vehicle price predictions for the re-seller's pricing system."

In [11]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='experiment_expectations', value=experiment_expectations)

---
## B. Feature Selection


In [12]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load data
try:
  X_train = pd.read_csv(at.folder_path / 'X_train.csv')
  y_train = pd.read_csv(at.folder_path / 'y_train.csv')

  X_val = pd.read_csv(at.folder_path / 'X_val.csv')
  y_val = pd.read_csv(at.folder_path / 'y_val.csv')

  X_test = pd.read_csv(at.folder_path / 'X_test.csv')
  y_test = pd.read_csv(at.folder_path / 'y_test.csv')
except Exception as e:
  print(e)

In [13]:
features_list = ["vehicle_brand","manufacturing_year","model_name","vehicle_type","engine_capacity","fuel_type","fuel_consumption","drive_type","transmission_type","kilometres_driven","engine_cylinders","doors","seats","territory","price"]

In [14]:
feature_selection_explanations = "The criteria I used for feature selections has 3 main conditions: first, it has to be consisten with both approaches, has to be relevant for the business problem (price) and I want to avoid not relevant or redundant variables. I chose to keep these main pricing features; manufacturing_year (vehicle age), engine_capacity (engine affects value), kilometres_driven (vehicle usage) and engine_cylinders (power). I also decided to include vehicle_brand (brand reputation), model_name (trust among users), vechicle_type (usually buyers look for a specific type according to their needs), fuel_type (preference), fuel_consumption (for efficiency), drive_type and transmission_type (driver's prefference). There are three variables that may not be too relevant, but could help find patterns would be doors (how practical the car is), seats (car's capacity) and territory (if there's a geographical factor affecting the prefference or budget). I decided to drop vehicle_colour as it doesn't seem to be important in none of the models, vehicle_condition as there are mainly used cars so it doesn't add much information. The other features were dropped because of privacy reasons and because they were overlapping with other features. Overall, this final feature set captures the most important aspects influencing vehicle resale prices, including vehicle age, technical specifications, usage, brand reputation, and potential regional differences."

In [15]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## C. Train Machine Learning Model

### C.1 Import Algorithm



In [16]:
from sklearn.linear_model import ElasticNet

In [17]:
algorithm_selection_explanations = "Elastic net is a regression algorithm, similar to linear regression but adds regularization. This step helps control how much influence each feature has in the model and prevents the model from relying too much on some variables. It combines two techniques, the first one is called L1 regularization (similar to Lasso) and L2 regularization (similar to Ridge). This algorithm is worth testing for this problem because many of the features in the dataset may be related to each other. For example, engine capacity, fuel consumption, and engine cylinders all describe aspects of the engine, and they may contain overlapping information. When features are correlated like this, a simple linear regression model can sometimes produce distorted results. Elastic net can help handle this situation by shrinking the coefficients of less important variables and focusing more on the features that are truly useful for predicting the price. From a business perspective, the goal of the project is to estimate the resale price of vehicles so the dealership can give customers a realistic price expectation when they bring a car to sell. If this model can produce slightly more stable and accurate predictions than the previous model, it could help provide more reliable price estimates and improve customer trust in the pricing system."

In [18]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='algorithm_selection_explanations', value=algorithm_selection_explanations)

In [19]:
X_train = pd.get_dummies(X_train, drop_first=True)
X_val = pd.get_dummies(X_val, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Fill missing numerical values with training median
X_train = X_train.fillna(X_train.median(numeric_only=True))
X_val = X_val.fillna(X_train.median(numeric_only=True))
X_test = X_test.fillna(X_train.median(numeric_only=True))

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

### C.2 Set Hyperparameters


In [20]:
# Elastic net (models with different hyperparameters)

elastic_model_1 = ElasticNet(alpha=0.15, l1_ratio=0.4, random_state=10)
elastic_model_2 = ElasticNet(alpha=0.8, l1_ratio=0.7, random_state=10)

In [21]:
# <Student to fill this section and then remove this comment>
hyperparameters_selection_explanations = "In this experiment, the hyperparameters alpha and l1_ratio were tuned in order to observe how different levels of regularization affect the performance of the model. The parameter alpha controls the strength of the regularization applied to the model. Smaller values apply weaker regularization, while larger values penalize the coefficients more strongly. In this experiment, two values were tested (0.15 and 0.8) in order to compare a configuration with relatively moderate regularization and another with stronger regularization. The parameter l1_ratio determines the balance between L1 and L2 regularization. L1 regularization can shrink some coefficients close to zero, which may help reduce the influence of less important variables, while L2 regularization tends to stabilize the model when features are correlated. Two values (0.4 and 0.7) were selected to test different mixes of these two effects. The first configuration places slightly more weight on L2 regularization, while the second configuration places more emphasis on the L1 component. These hyperparameters are worth testing because several variables in the dataset describe related features of the car, like engine characteristics or fuel consumption. Regularization could help the model focus on the most useful features and reduce the effect of redundant information. From a business perspective, testing different levels of regularization may help improve the reliability of the price estimation model. If elastic net produces more stable predictions than the basic linear regression model, the dealership could provide customers with more consistent price estimates when they bring their vehicles for resale."

In [22]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='hyperparameters_selection_explanations', value=hyperparameters_selection_explanations)

### C.3 Fit Model

In [23]:
# Elastic net model 1
elastic_model_1.fit(X_train_scaled, y_train)

# Elastic net model 2
elastic_model_2.fit(X_train_scaled, y_train)

ElasticNet(alpha=0.8, l1_ratio=0.7, random_state=10)

---
## D. Model Evaluation

### D.1 Model Technical Performance

In [24]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Predictions
y_pred_enet_1 = elastic_model_1.predict(X_val_scaled)
y_pred_enet_2 = elastic_model_2.predict(X_val_scaled)

# Metrics for model 1
mae_enet_1 = mean_absolute_error(y_val, y_pred_enet_1)
rmse_enet_1 = np.sqrt(mean_squared_error(y_val, y_pred_enet_1))
r2_enet_1 = r2_score(y_val, y_pred_enet_1)

print("Elastic Net Model 1")
print("MAE:", mae_enet_1)
print("RMSE:", rmse_enet_1)
print("R2:", r2_enet_1)

print("\n")

# Metrics for model 2
mae_enet_2 = mean_absolute_error(y_val, y_pred_enet_2)
rmse_enet_2 = np.sqrt(mean_squared_error(y_val, y_pred_enet_2))
r2_enet_2 = r2_score(y_val, y_pred_enet_2)

print("Elastic Net Model 2")
print("MAE:", mae_enet_2)
print("RMSE:", rmse_enet_2)
print("R2:", r2_enet_2)

Elastic Net Model 1
MAE: 10181.432713558883
RMSE: 22086.93243821241
R2: 0.5329743860528935


Elastic Net Model 2
MAE: 11217.226978814795
RMSE: 23316.616261384945
R2: 0.4795237161474769


In [25]:
# <Student to fill this section and then remove this comment>
model_performance_explanations = "These results show model 1 performs better than model 2 across all evaluation metrics. Model 1 achieved a MAE of about $10,181, which means that on average the predicted price differs from the real vehicle price by around ten thousand dollars. The RMSE is about $22,087, which indicates that some prediction errors are larger, since RMSE penalizes larger mistakes more strongly. The R2 value is 0.53, meaning that the model is able to explain approximately 53% of the variation in vehicle prices in the validation dataset. Alternatively, model 2 performs slightly worse. It has a higher MAE of around $11,217 and a higher RMSE of about $23,317, which indicates larger prediction errors compared to Model 1. Its R2 is 0.48, meaning that it explains about 48% of the variation in vehicle prices, which is lower than the first model. This suggests that the first Elastic Net configuration (alpha = 0.2, l1_ratio = 0.6) provides a better balance between L1 and L2 regularization for this dataset. The stronger regularization used in Model 2 appears to reduce the model’s ability to capture the relationships between vehicle features and resale price. When comparing these results with the linear regression model from experiment 1, elastic net performs slightly worse. The previous model achieved an R2 of about 0.57 and lower prediction errors, meaning it was able to explain more of the variation in vehicle prices. This suggests that the dataset may not suffer strongly from multicollinearity or overfitting, which are problems elastic net is designed to address. Summarizing, elastic net still produces reasonable predictions, but in this case the simpler linear regression model seems to provide better performance for predicting vehicle resale prices."

In [26]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='model_performance_explanations', value=model_performance_explanations)

### D.2 Business Impact from Current Model Performance


In [27]:
y_val_array = y_val.values.ravel()

# Calculate prediction errors
errors = np.abs(y_val_array - y_pred_enet_1)

# Percentages
within_5k = np.mean(errors <= 5000) * 100
within_10k = np.mean(errors <= 10000) * 100
within_20k = np.mean(errors <= 20000) * 100

print("Predictions within $5,000:", within_5k, "%")
print("Predictions within $10,000:", within_10k, "%")
print("Predictions within $20,000:", within_20k, "%")

Predictions within $5,000: 50.42081101759756 %
Predictions within $10,000: 70.61973986228003 %
Predictions within $20,000: 88.56159143075746 %


In [28]:
# <Student to fill this section and then remove this comment>
business_impacts_explanations = "The results show that about 50% of the predicted prices are within $5,000 of the real vehicle price. This means that for around half of the vehicles, the model can provide a fairly close estimate of price. From a business perspective, this level of accuracy could already be useful when giving customers an initial idea of how much their car could be worth. When the error margin increases to $10,000, the percentage of accurate predictions rises to about 70.6%. This means that for most vehicles, the model provides a reasonable price estimate that is relatively close to the real selling price. For a car reseller, this could still be useful for guiding pricing decisions or starting negotiations with customers. Finally, around 88.6% of the predictions fall within $20,000 of the actual price. This indicates that the model is able to provide at least a general estimate for most vehicles in the dataset. However, for higher value cars, an error of $20,000 could still be significant and might lead to incorrect pricing suggestions. From a business perspective, these results suggest that the model can already provide useful price guidance, but it is not perfectly accurate. Incorrect predictions may affect the business in different ways. If the predicted price is too low, the seller may feel that the estimated value is unfair and may decide not to list the car through the platform. On the other hand, if the predicted price is too high, the reseller might list the vehicle at an unrealistic price, which could make the car harder to sell and increase the time it stays in inventory. Overall, the model can already support the business goal of providing price estimates to customers, but there is still room for improvement. Improving prediction accuracy could increase customer trust and help the reseller set more competitive prices, which may lead to faster sales and better business performance."

In [29]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='business_impacts_explanations', value=business_impacts_explanations)

## E. Conclusion

In [30]:
# <Student to fill this section and then remove this comment>
experiment_outcome = "Hypothesis Partially Confirmed"

In [31]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h2", key='experiment_outcomes_explanations', value=experiment_outcome)

In [32]:
# <Student to fill this section and then remove this comment>
experiment_results_explanations = "I expected that this model would potentially improved the prediction performance compared to the previous linear regression model by introducing regularization and reducing potential overfitting. However, the results show that it didn’t. When focusing on the RMSE metric, which is the most relevant metric for this project, this model achieved an RMSE of around 22,087. This means that the typical prediction error is about $22,000 dollars. Although the model still produces useful estimates, this error is slightly higher than the RMSE obtained with the linear regression model in the first experiment. Because RMSE penalizes large prediction errors more strongly, it is the most important metric for evaluating the practical usefulness of the model in the business context. One insight from this experiment is that regularization does not appear to significantly improve model performance for this dataset. This may indicate that the dataset does not suffer strongly from multicollinearity or overfitting, which are problems this model usually addresses. Despite this, the experiment still provides useful information. It confirms that the relationship between the selected vehicle features and the resale price can already be captured reasonably well using a simpler linear model. This is why continuing experimentation is still worthwhile, but it may be more beneficial to explore non-linear models that can capture more complex relationships between variables. The resale price of vehicles may depend on interactions between factors such as brand, model, engine characteristics, and mileage, which linear models may not fully capture. Based on the results obtained so far, the following potential next experiments can be considered, as the KNN model, that could potentially capture non-linear relationships between vehicle characteristics and price. It may improve prediction accuracy by comparing similar vehicles in the dataset. This experiment is expected to potentially reduce RMSE if similar vehicles have consistent pricing patterns. Maybe we could create new features such as vehicle age groups or mileage categories may help the models capture pricing patterns more clearly. This could slightly improve prediction performance. If the model performance improves in future experiments, the final step would be to deploy the best performing model into the business system. In practice, this would involve integrating the trained model into the reseller’s platform so that when a customer submits vehicle information, the system automatically generates a price estimate based on the learned patterns in the data. This would allow the reseller to provide quick and data-driven price estimates to customers, improving customer trust and potentially increasing the number of vehicles listed through the business."

In [33]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h2", key='experiment_results_explanations', value=experiment_results_explanations)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=fbbefce8-41ae-47c6-bc64-96decd566c0b' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>